# Crack Binary Segmentation — Training

Notebook เดียวครบทั้ง pipeline การ train: config → dataset → augmentation → loader → sanity check → model → loss → metrics → optimizer → train loop → validation analysis → final test

- Dataset: `final_merged_dataset_v1/manifest_v2.csv` (ใช้คอลัมน์ `split` ไม่ใช่ชื่อโฟลเดอร์)
- ปรับทุกอย่างบน **validation** เท่านั้น — **Final Test** ปิดไว้ด้วย `RUN_TEST = False`
- แผนรวม: `plan.md`

## 1. Setup

In [ ]:
import os, json, math, random, time, sys
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
from skimage.morphology import skeletonize
from tqdm.auto import tqdm

cv2.setNumThreads(0)  # avoid oversubscription inside DataLoader workers


def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


DEVICE = get_device()
print("device:", DEVICE, "| torch", torch.__version__, "| smp", smp.__version__, "| albumentations", A.__version__)

## 2. Config
ค่าทั้งหมดอยู่ที่นี่ที่เดียว — ถูกบันทึกเป็น `config.json` ในโฟลเดอร์ run

In [ ]:
ROOT = Path("/Users/attachatchannakorn/Documents/crack-binary-segmentation")

CFG = dict(
    # data
    dataset_dir=str(ROOT / "Dataset/final_merged_dataset_v1"),
    manifest="manifest_v2.csv",
    runs_dir=str(ROOT / "runs"),
    img_size=448,
    subset=None,            # e.g. 0.05 = use 5% of every split for a quick debug run
    # model
    model_name="unet",
    encoder="resnet34",
    encoder_weights="imagenet",
    # training
    epochs=40,
    batch_size=8,
    # macOS: forked DataLoader workers deadlock once torch/MPS threads exist, and spawn cannot
    # pickle classes defined in a notebook -> load in the main process (costs only ~5% on M2).
    num_workers=0 if DEVICE.type == "mps" else 4,
    lr_encoder=1e-4,
    lr_decoder=1e-3,
    weight_decay=1e-4,
    warmup_epochs=1,
    grad_clip=1.0,
    amp=DEVICE.type == "cuda",   # fp16 autocast on MPS measured slower than fp32 (2.27 vs 1.86 s/step)
    # loss
    bce_weight=0.5,
    dice_weight=0.5,
    pos_weight=None,        # e.g. 3.0 if recall stays low
    # evaluation
    threshold=0.5,          # used during training; tuned on validation in section 14
    tol_px=2,               # tolerance for tol-F1
    monitor="tol_f1",       # checkpoint selection metric (validation)
    early_stop_patience=10,
    resume_from=None,       # path to a run's last.pt to continue training
    seed=42,
)

RUN_OVERFIT_TEST = True   # section 10: overfit one batch to catch pipeline bugs
RUN_TEST = False          # section 15: flip to True ONCE, after model/threshold/TTA are frozen

RUN_NAME = f"{datetime.now():%Y%m%d_%H%M%S}_{CFG['model_name']}_{CFG['encoder']}"
RUN_DIR = Path(CFG["runs_dir"]) / RUN_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
(RUN_DIR / "config.json").write_text(json.dumps(CFG, indent=2))
seed_everything(CFG["seed"])
print("run dir:", RUN_DIR)

## 3. Manifest & splits

In [ ]:
DATA_DIR = Path(CFG["dataset_dir"])
df = pd.read_csv(DATA_DIR / CFG["manifest"])

# Split comes from the `split` column, NOT the folder name: manifest_v2 re-split some UAV images.
SPLITS = ["train", "validation", "test"]
frames = {s: df[df.split == s].reset_index(drop=True) for s in SPLITS}
if CFG["subset"]:
    frames = {s: f.sample(frac=CFG["subset"], random_state=CFG["seed"]).reset_index(drop=True)
              for s, f in frames.items()}

print({s: len(f) for s, f in frames.items()})
_all = pd.concat(frames.values(), ignore_index=True)
pd.crosstab(_all.crack_area_class, _all.split, margins=True)

## 4. Dataset class

In [ ]:
class CrackDataset(Dataset):
    def __init__(self, frame, root, transform):
        self.images = [str(root / p) for p in frame.image_path]
        self.masks = [str(root / p) for p in frame.mask_path]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        img = cv2.imread(self.images[i], cv2.IMREAD_COLOR)  # also drops alpha of the 4 RGBA PNGs
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = (cv2.imread(self.masks[i], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
        out = self.transform(image=img, mask=mask)
        return out["image"], out["mask"].unsqueeze(0).float(), i

## 5. Augmentation
- train: geometric + photometric + **จำลองภาพโดรน** (blur, downscale, JPEG, noise, shadow)
- val/test: normalize อย่างเดียว
- mask ใช้ nearest interpolation (ค่าเริ่มต้นของ albumentations) → ยังเป็น 0/1

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)  # ImageNet
S = CFG["img_size"]

train_tf = A.Compose([
    A.RandomResizedCrop(size=(S, S), scale=(0.5, 1.0), ratio=(0.75, 1.33), p=0.5),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.RandomRotate90(),
    A.Affine(scale=(0.9, 1.1), translate_percent=(-0.05, 0.05), rotate=(-15, 15),
             border_mode=cv2.BORDER_REFLECT_101, p=0.3),
    # photometric
    A.RandomBrightnessContrast(p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.3),
    A.CLAHE(p=0.1),
    # drone-like degradations
    A.OneOf([A.MotionBlur(blur_limit=(3, 7)), A.GaussianBlur(blur_limit=(3, 5))], p=0.2),
    A.Downscale(scale_range=(0.5, 0.9), p=0.1),
    A.ImageCompression(quality_range=(50, 95), p=0.2),
    A.GaussNoise(std_range=(0.02, 0.06), p=0.15),
    A.RandomShadow(p=0.1),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

eval_tf = A.Compose([A.Normalize(mean=MEAN, std=STD), ToTensorV2()])

## 6. DataLoader
บน Mac ใช้ `num_workers=0` (worker แบบ fork ค้างบน macOS) — วัดแล้วโหลดข้อมูลใช้เวลาแค่ ~5% ของ train step บน M2

In [ ]:
datasets = {
    "train": CrackDataset(frames["train"], DATA_DIR, train_tf),
    "validation": CrackDataset(frames["validation"], DATA_DIR, eval_tf),
    "test": CrackDataset(frames["test"], DATA_DIR, eval_tf),
}


def worker_init_fn(_):
    # every forked worker starts with the parent's RNG state -> reseed so augmentations differ
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)
    torch.utils.data.get_worker_info().dataset.transform.set_random_seed(seed)


def make_loader(split, shuffle):
    nw = CFG["num_workers"]
    return DataLoader(
        datasets[split],
        batch_size=CFG["batch_size"],
        shuffle=shuffle,
        drop_last=shuffle,
        num_workers=nw,
        pin_memory=DEVICE.type == "cuda",
        persistent_workers=nw > 0,
        worker_init_fn=worker_init_fn if nw > 0 else None,
    )


loaders = {
    "train": make_loader("train", shuffle=True),
    "validation": make_loader("validation", shuffle=False),
    "test": make_loader("test", shuffle=False),
}
print({s: len(l) for s, l in loaders.items()}, "batches")

## 7. Sanity check — data

In [ ]:
xb, yb, ib = next(iter(loaders["train"]))
print("image", tuple(xb.shape), xb.dtype, f"range [{xb.min():.2f}, {xb.max():.2f}]")
print("mask ", tuple(yb.shape), yb.dtype, "values", yb.unique().tolist(), f"crack ratio {yb.mean():.4f}")
assert xb.shape[1:] == (3, S, S) and yb.shape[1:] == (1, S, S)
assert set(yb.unique().tolist()) <= {0.0, 1.0}

xv, yv, _ = next(iter(loaders["validation"]))
assert set(yv.unique().tolist()) <= {0.0, 1.0}

t0 = time.time()
for k, _ in zip(range(20), loaders["train"]):
    pass
print(f"loader throughput: {20 * CFG['batch_size'] / (time.time() - t0):.1f} img/s")


def denorm(x):
    x = x.permute(1, 2, 0).cpu().numpy() * STD + MEAN
    return np.clip(x, 0, 1)


def overlay(img, mask, color=(1, 0, 0)):
    out = img.copy()
    out[mask > 0.5] = color
    return out


n = min(8, len(xb))
fig, axes = plt.subplots(2, n, figsize=(2.2 * n, 4.6))
for j in range(n):
    name = frames["train"].sample_id[int(ib[j])]
    axes[0, j].imshow(denorm(xb[j])); axes[0, j].set_title(name[:18], fontsize=7)
    axes[1, j].imshow(overlay(denorm(xb[j]), yb[j, 0].numpy()))
for a in axes.ravel():
    a.axis("off")
plt.suptitle("train batch after augmentation (bottom: mask overlay)")
plt.tight_layout(); plt.show()

## 8. Model

In [ ]:
def build_model():
    if CFG["model_name"] == "unet":
        m = smp.Unet(CFG["encoder"], encoder_weights=CFG["encoder_weights"], classes=1)
    else:
        raise NotImplementedError(f"model_name={CFG['model_name']!r} (ConvNeXt-UPerNet comes next)")
    return m.to(DEVICE)


model = build_model()
n_params = sum(p.numel() for p in model.parameters())
print(f"{CFG['model_name']}-{CFG['encoder']}: {n_params / 1e6:.2f}M params")

model.eval()
with torch.no_grad():
    out = model(xb[:2].to(DEVICE))
print("output", tuple(out.shape), "(logits)")
assert out.shape == (2, 1, S, S)

## 9. Loss & Metrics
- **Loss** = BCE + (1 − Dice) โดย Dice คิดระดับ **batch** (รวมทั้ง batch) กัน 0/0 จากรูปที่ mask ว่าง
- **Metrics** สะสม TP/FP/FN ทั้ง dataset (micro): IoU, Dice (= pixel F1), Precision, Recall
- **tol-F1**: ยอมคลาดได้ `tol_px` พิกเซล (Chebyshev) — pred ที่อยู่ใกล้ GT นับถูก, GT ที่มี pred อยู่ใกล้นับเจอ

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5, pos_weight=None, eps=1.0):
        super().__init__()
        pw = torch.tensor([pos_weight]) if pos_weight else None
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pw)
        self.bw, self.dw, self.eps = bce_weight, dice_weight, eps

    def forward(self, logits, target):
        bce = self.bce(logits, target)
        p = torch.sigmoid(logits)
        dice = (2 * (p * target).sum() + self.eps) / (p.sum() + target.sum() + self.eps)
        return self.bw * bce + self.dw * (1 - dice)


criterion = BCEDiceLoss(CFG["bce_weight"], CFG["dice_weight"], CFG["pos_weight"]).to(DEVICE)

COUNT_KEYS = ["tp", "fp", "fn", "pred_near_gt", "gt_near_pred"]


def dilate(m, r):
    return F.max_pool2d(m, 2 * r + 1, stride=1, padding=r) if r > 0 else m


@torch.no_grad()
def batch_counts(prob, target, thr, tol):
    """Per-image pixel counts, shape (N, 5) in COUNT_KEYS order."""
    pred = (prob > thr).float()
    dims = (1, 2, 3)
    tp = (pred * target).sum(dims)
    fp = (pred * (1 - target)).sum(dims)
    fn = ((1 - pred) * target).sum(dims)
    pred_near_gt = (pred * dilate(target, tol)).sum(dims)
    gt_near_pred = (target * dilate(pred, tol)).sum(dims)
    return torch.stack([tp, fp, fn, pred_near_gt, gt_near_pred], 1)


def summarize(c):
    tp, fp, fn, png, gnp = [float(v) for v in c]
    eps = 1e-9
    p, r = tp / (tp + fp + eps), tp / (tp + fn + eps)
    tp_, tr_ = png / (tp + fp + eps), gnp / (tp + fn + eps)
    return dict(
        iou=tp / (tp + fp + fn + eps),
        dice=2 * tp / (2 * tp + fp + fn + eps),
        precision=p,
        recall=r,
        tol_precision=tp_,
        tol_recall=tr_,
        tol_f1=2 * tp_ * tr_ / (tp_ + tr_ + eps),
    )


# quick self-check: perfect prediction -> all metrics 1
_c = batch_counts(yb.to(DEVICE), yb.to(DEVICE), 0.5, CFG["tol_px"]).sum(0).cpu()
print({k: round(v, 4) for k, v in summarize(_c).items()})

## 10. Sanity check — overfit 1 batch
ถ้า pipeline ถูก loss ต้องลดลงชัดเจนภายในไม่กี่สิบ step (ใช้โมเดลแยก ไม่กระทบโมเดลจริง)

In [ ]:
if RUN_OVERFIT_TEST:
    m = build_model()
    opt = torch.optim.AdamW(m.parameters(), lr=1e-3)
    x_, y_ = xb.to(DEVICE), yb.to(DEVICE)
    m.train()
    losses = []
    for step in range(60):
        opt.zero_grad(set_to_none=True)
        loss = criterion(m(x_), y_)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    print("loss:", " ".join(f"{l:.3f}" for l in losses[::10]), f"-> {losses[-1]:.3f}")
    m.eval()
    with torch.no_grad():
        print("train-batch metrics:", {k: round(v, 3) for k, v in summarize(
            batch_counts(torch.sigmoid(m(x_)), y_, 0.5, CFG["tol_px"]).sum(0).cpu()).items()})
    assert losses[-1] < 0.5 * losses[0], "loss did not drop — check data/model/loss"
    del m, opt

## 11. Optimizer & Scheduler
AdamW แยก lr encoder (pretrained, ต่ำ) กับ decoder (สูง) + linear warmup → cosine (step ทุก batch)

In [ ]:
model = build_model()

enc_params = list(model.encoder.parameters())
enc_ids = {id(p) for p in enc_params}
dec_params = [p for p in model.parameters() if id(p) not in enc_ids]
optimizer = torch.optim.AdamW(
    [{"params": enc_params, "lr": CFG["lr_encoder"]},
     {"params": dec_params, "lr": CFG["lr_decoder"]}],
    weight_decay=CFG["weight_decay"],
)

steps_per_epoch = len(loaders["train"])
total_steps = CFG["epochs"] * steps_per_epoch
warmup_steps = CFG["warmup_epochs"] * steps_per_epoch


def lr_lambda(step):
    if step < warmup_steps:
        return (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1 + math.cos(math.pi * min(1.0, progress)))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda")

start_epoch, history, best_score = 1, [], -1.0
if CFG["resume_from"]:
    ck = torch.load(CFG["resume_from"], map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"]); optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"]); scaler.load_state_dict(ck["scaler"])
    start_epoch, history, best_score = ck["epoch"] + 1, ck["history"], ck["best_score"]
    print(f"resumed from {CFG['resume_from']} at epoch {start_epoch}")

print(f"{steps_per_epoch} steps/epoch, {total_steps} total, warmup {warmup_steps}")

## 12. Train loop
บันทึก `history.csv`, `last.pt` (resume ได้) และ `best.pt` (ตาม `monitor` บน validation) ทุก epoch

In [ ]:
def autocast():
    return torch.autocast(DEVICE.type, dtype=torch.float16, enabled=CFG["amp"])


def train_one_epoch(epoch):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loaders["train"], desc=f"epoch {epoch} train", leave=False)
    for x, y, _ in pbar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(x)
        loss = criterion(logits.float(), y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total += loss.item() * len(x)
        n += len(x)
        pbar.set_postfix(loss=f"{total / n:.4f}")
    return total / n


@torch.no_grad()
def evaluate(loader, thr):
    model.eval()
    total, n, counts = 0.0, 0, []
    for x, y, _ in tqdm(loader, desc="eval", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        with autocast():
            logits = model(x)
        logits = logits.float()
        total += criterion(logits, y).item() * len(x)
        n += len(x)
        counts.append(batch_counts(torch.sigmoid(logits), y, thr, CFG["tol_px"]).cpu())
    return {"loss": total / n, **summarize(torch.cat(counts).sum(0))}


bad_epochs = 0
for epoch in range(start_epoch, CFG["epochs"] + 1):
    t0 = time.time()
    train_loss = train_one_epoch(epoch)
    val = evaluate(loaders["validation"], CFG["threshold"])
    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val.items()},
           "lr_decoder": optimizer.param_groups[1]["lr"], "time_s": time.time() - t0}
    history.append(row)
    pd.DataFrame(history).to_csv(RUN_DIR / "history.csv", index=False)

    score = val[CFG["monitor"]]
    improved = score > best_score
    if improved:
        best_score, bad_epochs = score, 0
    else:
        bad_epochs += 1
    print(f"[{epoch:3d}] train {train_loss:.4f} | val loss {val['loss']:.4f} iou {val['iou']:.4f} "
          f"dice {val['dice']:.4f} P {val['precision']:.3f} R {val['recall']:.3f} "
          f"tolF1 {val['tol_f1']:.4f} | {row['time_s'] / 60:.1f} min" + ("  * best" if improved else ""))

    ckpt = {"model": model.state_dict(), "epoch": epoch, "cfg": CFG, "val": val,
            "history": history, "best_score": best_score}
    if improved:
        torch.save(ckpt, RUN_DIR / "best.pt")
    ckpt.update(optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(), scaler=scaler.state_dict())
    torch.save(ckpt, RUN_DIR / "last.pt")

    if bad_epochs >= CFG["early_stop_patience"]:
        print(f"early stop: no {CFG['monitor']} improvement for {bad_epochs} epochs")
        break

## 13. Training curves

In [ ]:
h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h.epoch, h.train_loss, label="train"); ax[0].plot(h.epoch, h.val_loss, label="val")
ax[0].set_title("loss"); ax[0].legend()
for k in ["iou", "dice", "tol_f1"]:
    ax[1].plot(h.epoch, h[f"val_{k}"], label=k)
ax[1].set_title("val metrics @ thr=%.2f" % CFG["threshold"]); ax[1].legend()
ax[2].plot(h.epoch, h.val_precision, label="precision"); ax[2].plot(h.epoch, h.val_recall, label="recall")
ax[2].set_title("val precision / recall"); ax[2].legend()
for a in ax:
    a.set_xlabel("epoch"); a.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(RUN_DIR / "curves.png", dpi=120); plt.show()

## 14. Validation analysis (ใช้ best.pt)
1. จูน threshold (0.05–0.95) ทั้งแบบไม่มี / มี TTA (flip)
2. แยกผลตาม `crack_area_class` และ `source_dataset`
3. False alarm บนรูป negative
4. clDice (ความต่อเนื่องของรอยร้าว)
5. ดูภาพที่โมเดลพลาดมากที่สุด

ผลที่เลือก (threshold, TTA) บันทึกลง `val_selection.json` เพื่อใช้กับ Final Test

In [ ]:
best = torch.load(RUN_DIR / "best.pt", map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"])
model.eval()
print(f"loaded best.pt from epoch {best['epoch']} ({CFG['monitor']}={best['val'][CFG['monitor']]:.4f} @ thr {CFG['threshold']})")

THRS = np.round(np.arange(0.05, 0.96, 0.05), 2)


@torch.no_grad()
def predict_prob(x, tta=False):
    with autocast():
        p = torch.sigmoid(model(x).float())
        if tta:
            for dims in ([-1], [-2], [-2, -1]):
                p = p + torch.sigmoid(model(torch.flip(x, dims)).float()).flip(dims)
            p = p / 4
    return p


@torch.no_grad()
def collect_counts(loader, tta=False):
    """Per-image counts for every threshold: array (N, len(THRS), 5), in dataset order."""
    out = []
    for x, y, _ in tqdm(loader, desc=f"predict tta={tta}", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        p = predict_prob(x, tta)
        out.append(torch.stack([batch_counts(p, y, t, CFG["tol_px"]) for t in THRS], 1).cpu())
    return torch.cat(out).numpy()


def threshold_table(C):
    return pd.DataFrame([{"thr": t, **summarize(C[:, i].sum(0))} for i, t in enumerate(THRS)])


val_frame = frames["validation"]
C_plain = collect_counts(loaders["validation"], tta=False)
C_tta = collect_counts(loaders["validation"], tta=True)
tab_plain, tab_tta = threshold_table(C_plain), threshold_table(C_tta)

mon = CFG["monitor"]
bp, bt = tab_plain.loc[tab_plain[mon].idxmax()], tab_tta.loc[tab_tta[mon].idxmax()]
USE_TTA = bool(bt[mon] > bp[mon] + 0.002)   # only keep TTA if it clearly helps (it costs 4x inference)
BEST_THR = float(bt.thr if USE_TTA else bp.thr)
C_best = (C_tta if USE_TTA else C_plain)[:, list(THRS).index(BEST_THR)]

print(f"best no-TTA: thr {bp.thr:.2f} {mon} {bp[mon]:.4f} | best TTA: thr {bt.thr:.2f} {mon} {bt[mon]:.4f}")
print(f"-> selected thr={BEST_THR:.2f}, TTA={USE_TTA}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, tab in [("no TTA", tab_plain), ("TTA", tab_tta)]:
    ax[0].plot(tab.thr, tab[mon], label=f"{mon} {name}"); ax[0].plot(tab.thr, tab.dice, "--", label=f"dice {name}")
    ax[1].plot(tab.recall, tab.precision, label=name)
ax[0].axvline(BEST_THR, color="k", lw=0.8); ax[0].set_xlabel("threshold"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("PR (pixel)"); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
tab_plain.round(4)

In [ ]:
# per crack-area class and per source, at the selected threshold / TTA
def grouped(frame, C, col):
    rows = []
    for g, idx in frame.groupby(col).groups.items():
        rows.append({col: g, "n": len(idx), **summarize(C[idx].sum(0))})
    return pd.DataFrame(rows).set_index(col)


metric_cols = ["n", "iou", "dice", "precision", "recall", "tol_f1"]
by_class = grouped(val_frame, C_best, "crack_area_class").loc[["small", "medium", "large", "xlarge"], metric_cols]
by_source = grouped(val_frame, C_best, "source_dataset")[metric_cols].sort_values("tol_f1")

neg = val_frame.crack_area_class.eq("negative").to_numpy()
fp_px = C_best[neg, 1]
neg_report = {
    "negative_images": int(neg.sum()),
    "any_fp_rate": float((fp_px > 0).mean()),
    "fp_ge_100px_rate": float((fp_px >= 100).mean()),
    "mean_fp_px": float(fp_px.mean()),
}
print("negatives:", neg_report)
display(by_class.round(4))
display(by_source.round(4))

In [ ]:
# clDice + worst cases need the actual binary predictions
@torch.no_grad()
def predict_masks(loader, thr, tta):
    preds = []
    for x, _, _ in tqdm(loader, desc="masks", leave=False):
        preds.append((predict_prob(x.to(DEVICE), tta) > thr).cpu().numpy()[:, 0])
    return np.concatenate(preds)


def cldice_counts(pred, gt):
    """Numerators/denominators of clDice (topology precision / sensitivity)."""
    sp, sg = skeletonize(pred), skeletonize(gt)
    return np.array([(sp & gt).sum(), sp.sum(), (sg & pred).sum(), sg.sum()], dtype=np.float64)


val_pred = predict_masks(loaders["validation"], BEST_THR, USE_TTA)
val_gt = np.stack([cv2.imread(str(DATA_DIR / p), cv2.IMREAD_GRAYSCALE) > 127 for p in val_frame.mask_path])

pos = np.nonzero(~neg)[0]
cl = np.sum([cldice_counts(val_pred[i], val_gt[i]) for i in tqdm(pos, desc="clDice", leave=False)], axis=0)
t_prec, t_sens = cl[0] / max(cl[1], 1), cl[2] / max(cl[3], 1)
cldice = 2 * t_prec * t_sens / max(t_prec + t_sens, 1e-9)
print(f"clDice {cldice:.4f} (topology precision {t_prec:.4f}, sensitivity {t_sens:.4f})")

val_summary = {"threshold": BEST_THR, "tta": USE_TTA, "epoch": best["epoch"],
               **summarize(C_best.sum(0)), "cldice": cldice, "negatives": neg_report}
(RUN_DIR / "val_selection.json").write_text(json.dumps(val_summary, indent=2))
by_class.to_csv(RUN_DIR / "val_by_class.csv"); by_source.to_csv(RUN_DIR / "val_by_source.csv")
{k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_summary.items()}

In [ ]:
# worst positives by per-image tol-F1
def per_image_tol_f1(C):
    tp, fp, fn, png, gnp = C.T
    p, r = png / np.maximum(tp + fp, 1), gnp / np.maximum(tp + fn, 1)
    return 2 * p * r / np.maximum(p + r, 1e-9)


f1_img = per_image_tol_f1(C_best)
worst = pos[np.argsort(f1_img[pos])[:6]]
fig, axes = plt.subplots(len(worst), 3, figsize=(9, 3 * len(worst)))
for row, i in zip(axes, worst):
    r = val_frame.iloc[i]
    img = cv2.cvtColor(cv2.imread(str(DATA_DIR / r.image_path)), cv2.COLOR_BGR2RGB) / 255.0
    row[0].imshow(img); row[0].set_title(f"{r.sample_id[:28]}\n{r.source_dataset} / {r.crack_area_class}", fontsize=8)
    row[1].imshow(overlay(img, val_gt[i], (0, 1, 0))); row[1].set_title("GT", fontsize=8)
    row[2].imshow(overlay(img, val_pred[i])); row[2].set_title(f"pred (tol-F1 {f1_img[i]:.2f})", fontsize=8)
    for a in row:
        a.axis("off")
plt.tight_layout(); plt.show()

## 15. Final Test — รันครั้งเดียว
ใช้ best.pt + threshold + TTA ที่เลือกจาก validation เท่านั้น ห้ามกลับไปปรับอะไรหลังเห็นผลนี้

In [ ]:
if RUN_TEST:
    sel = json.loads((RUN_DIR / "val_selection.json").read_text())
    test_frame = frames["test"]
    idx = list(THRS).index(sel["threshold"])
    C_test = collect_counts(loaders["test"], tta=sel["tta"])[:, idx]
    test_pred = predict_masks(loaders["test"], sel["threshold"], sel["tta"])
    test_gt = np.stack([cv2.imread(str(DATA_DIR / p), cv2.IMREAD_GRAYSCALE) > 127 for p in test_frame.mask_path])
    tneg = test_frame.crack_area_class.eq("negative").to_numpy()
    cl = np.sum([cldice_counts(test_pred[i], test_gt[i]) for i in np.nonzero(~tneg)[0]], axis=0)
    tp_, ts_ = cl[0] / max(cl[1], 1), cl[2] / max(cl[3], 1)
    test_summary = {**sel, **summarize(C_test.sum(0)), "cldice": 2 * tp_ * ts_ / max(tp_ + ts_, 1e-9),
                    "any_fp_rate_negatives": float((C_test[tneg, 1] > 0).mean())}
    test_summary.pop("negatives", None)
    (RUN_DIR / "test_results.json").write_text(json.dumps(test_summary, indent=2))
    grouped(test_frame, C_test, "crack_area_class").to_csv(RUN_DIR / "test_by_class.csv")
    grouped(test_frame, C_test, "source_dataset").to_csv(RUN_DIR / "test_by_source.csv")
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in test_summary.items()})
else:
    print("RUN_TEST=False — skipped")